# Final Evaluation – XGBoost


## 1. Zielsetzung und methodische Einordnung

Nach Abschluss der Modellauswahl und Modellinterpretation wird das festgelegte getunte XGBoost-Modell abschließend auf dem bislang unangetasteten Testdatensatz August 2026 evaluiert. Hierzu werden Training und Validation aus den Monaten November 2025 bis Juli 2026 zu einem finalen Trainingsdatensatz zusammengeführt. Sämtliche Preprocessing-Schritte werden ausschließlich auf diesem finalen Trainingsdatensatz neu gefittet. Die Hyperparameter und Features bleiben unverändert. Die Ergebnisse des August-Testdatensatzes werden nicht für weitere Modellentscheidungen verwendet.

Alle drei Splits basieren auf der aktualisierten temporalen Split-Logik. Fahrten, deren `concrete_ride_id` über eine Split-Grenze läuft, wurden vor der Feature-Erstellung aus den Modellierungsdaten ausgeschlossen und werden nicht in Training, Validation oder Test verwendet.

## 2. Setup


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor


PROJECT_DIR = Path.cwd()

while not (PROJECT_DIR / "datasets").exists():
    if PROJECT_DIR == PROJECT_DIR.parent:
        raise FileNotFoundError("Could not find project directory with datasets/.")
    PROJECT_DIR = PROJECT_DIR.parent

FEATURE_DIR = PROJECT_DIR / "datasets" / "modeling" / "features"
TRAIN_PATH = FEATURE_DIR / "train_features.parquet"
VALIDATION_PATH = FEATURE_DIR / "validation_features.parquet"
TEST_PATH = FEATURE_DIR / "test_features.parquet"

TRAIN_PERIOD = "November 2025 bis Juni 2026"
VALIDATION_PERIOD = "Juli 2026"
TEST_PERIOD = "August 2026"
FINAL_TRAIN_PERIOD = "November 2025 bis Juli 2026"

## 3. Daten laden


In [ ]:
train = pd.read_parquet(TRAIN_PATH)
validation = pd.read_parquet(VALIDATION_PATH)
test = pd.read_parquet(TEST_PATH)

dataset_overview = pd.DataFrame(
    {
        "Dataset": ["Training", "Validation", "Test"],
        "Period": [TRAIN_PERIOD, VALIDATION_PERIOD, TEST_PERIOD],
        "Observations": [len(train), len(validation), len(test)],
    }
)

dataset_overview

## 4. Finalen Trainingsdatensatz November 2025 bis Juli 2026 erstellen

Nach Abschluss aller Modellierungsentscheidungen darf der bisherige Validierungsdatensatz in das finale Training einbezogen werden. Der August-Testdatensatz bleibt davon vollständig getrennt. Die Boundary-Rides sind zu diesem Zeitpunkt bereits aus beiden finalen Trainingsanteilen und aus dem Testdatensatz entfernt.

In [ ]:
final_train = pd.concat(
    [train, validation],
    ignore_index=True,
)

y_final_train = final_train["delay_target"]
y_test = test["delay_target"]

final_dataset_overview = pd.DataFrame(
    {
        "Dataset": ["Final Training", "Final Test"],
        "Period": [FINAL_TRAIN_PERIOD, TEST_PERIOD],
        "Observations": [len(final_train), len(test)],
    }
)

final_dataset_overview

## 5. Features und Preprocessing


In [ ]:
external_features = [
    "event_hour",
    "event_weekday",
    "is_public_holiday",
    "ff_wind_speed_ms",
    "wind_direction_sin",
    "wind_direction_cos",
    "fx_wind_gust_ms",
    "tu_temperature_c",
    "tu_relative_humidity_pct",
    "rr_precipitation_mm",
    "rr_precipitation_indicator",
    "rr_precipitation_form",
]

X_final_train = final_train[external_features].copy()
X_test = test[external_features].copy()

categorical_features = [
    "event_hour",
    "event_weekday",
    "rr_precipitation_form",
]

numeric_features = [
    "is_public_holiday",
    "ff_wind_speed_ms",
    "wind_direction_sin",
    "wind_direction_cos",
    "fx_wind_gust_ms",
    "tu_temperature_c",
    "tu_relative_humidity_pct",
    "rr_precipitation_mm",
    "rr_precipitation_indicator",
]

model_features = categorical_features + numeric_features

X_final_train_model = X_final_train[model_features].copy()
X_test_model = X_test[model_features].copy()

for column in categorical_features:
    X_final_train_model[column] = X_final_train_model[column].fillna("missing").astype(str)
    X_test_model[column] = X_test_model[column].fillna("missing").astype(str)

numeric_transformer = SimpleImputer(strategy="median")
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)

feature_overview = pd.DataFrame(
    {
        "Feature": model_features,
        "Feature Type": [
            "categorical" if feature in categorical_features else "numeric"
            for feature in model_features
        ],
    }
)

feature_overview


## 6. Baselines auf November 2025 bis Juli 2026 fitten


In [ ]:
X_final_train_dummy = np.zeros((len(final_train), 1))
X_test_dummy = np.zeros((len(test), 1))

median_baseline_model = DummyRegressor(strategy="median")
median_baseline_model.fit(X_final_train_dummy, y_final_train)
median_baseline_test_predictions = median_baseline_model.predict(X_test_dummy)
median_baseline_prediction = float(median_baseline_model.constant_.ravel()[0])

median_baseline_test_mae = mean_absolute_error(
    y_test,
    median_baseline_test_predictions,
)
median_baseline_test_rmse = np.sqrt(
    mean_squared_error(y_test, median_baseline_test_predictions)
)
median_baseline_test_r2 = r2_score(y_test, median_baseline_test_predictions)

mean_baseline_model = DummyRegressor(strategy="mean")
mean_baseline_model.fit(X_final_train_dummy, y_final_train)
mean_baseline_test_predictions = mean_baseline_model.predict(X_test_dummy)
mean_baseline_prediction = float(mean_baseline_model.constant_.ravel()[0])

mean_baseline_test_mae = mean_absolute_error(
    y_test,
    mean_baseline_test_predictions,
)
mean_baseline_test_rmse = np.sqrt(
    mean_squared_error(y_test, mean_baseline_test_predictions)
)
mean_baseline_test_r2 = r2_score(y_test, mean_baseline_test_predictions)

baseline_predictions = pd.DataFrame(
    {
        "Baseline": ["Median Baseline", "Mean Baseline"],
        "Constant Prediction": [
            median_baseline_prediction,
            mean_baseline_prediction,
        ],
    }
)

baseline_predictions.round(3)


## 7. Finales XGBoost-Modell trainieren


In [ ]:
xgboost_final_params = {
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}

final_xgboost_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(**xgboost_final_params)),
    ]
)

final_xgboost_pipeline.fit(
    X_final_train_model,
    y_final_train,
)

xgboost_final_params


## 8. Finale Evaluation auf August


In [ ]:
final_test_predictions = final_xgboost_pipeline.predict(X_test_model)

final_test_mae = mean_absolute_error(y_test, final_test_predictions)
final_test_rmse = np.sqrt(mean_squared_error(y_test, final_test_predictions))
final_test_r2 = r2_score(y_test, final_test_predictions)

final_test_comparison = pd.DataFrame(
    [
        {
            "Model": "Median Baseline",
            "MAE": median_baseline_test_mae,
            "RMSE": median_baseline_test_rmse,
            "R2": median_baseline_test_r2,
        },
        {
            "Model": "Mean Baseline",
            "MAE": mean_baseline_test_mae,
            "RMSE": mean_baseline_test_rmse,
            "R2": mean_baseline_test_r2,
        },
        {
            "Model": "XGBoost Tuned",
            "MAE": final_test_mae,
            "RMSE": final_test_rmse,
            "R2": final_test_r2,
        },
    ]
)

final_test_comparison.round(3)


In [ ]:
final_train_predictions = final_xgboost_pipeline.predict(X_final_train_model)

final_train_mae = mean_absolute_error(y_final_train, final_train_predictions)
final_train_rmse = np.sqrt(mean_squared_error(y_final_train, final_train_predictions))
final_train_r2 = r2_score(y_final_train, final_train_predictions)

final_training_test_comparison = pd.DataFrame(
    [
        {
            "Dataset": "Final Training (Nov 2025–Jul 2026)",
            "MAE": final_train_mae,
            "RMSE": final_train_rmse,
            "R2": final_train_r2,
        },
        {
            "Dataset": "Final Test (August)",
            "MAE": final_test_mae,
            "RMSE": final_test_rmse,
            "R2": final_test_r2,
        },
    ]
)

final_training_test_comparison.round(3)


## 9. Fehler- und Vorhersageübersicht


In [ ]:
test_residuals = y_test - final_test_predictions
test_absolute_errors = np.abs(test_residuals)

error_overview = pd.DataFrame(
    {
        "Metric": [
            "Mean Residual",
            "Median Residual",
            "Mean Absolute Error",
            "Median Absolute Error",
            "95th Percentile Absolute Error",
            "Maximum Absolute Error",
        ],
        "Value": [
            np.mean(test_residuals),
            np.median(test_residuals),
            np.mean(test_absolute_errors),
            np.median(test_absolute_errors),
            np.percentile(test_absolute_errors, 95),
            np.max(test_absolute_errors),
        ],
    }
)

error_overview.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(test_residuals, bins=50)
ax.set_title("Verteilung der Prognosefehler im Testmonat August")
ax.set_xlabel("Residuum: Ist - Prognose (Minuten)")
ax.set_ylabel("Beobachtungen")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


Die folgende Grafik vergleicht die tatsächlichen August-Verspätungen mit den Modellvorhersagen. Punkte auf der diagonalen Linie entsprechen einer perfekten Prognose. Punkte unterhalb der Linie zeigen, dass XGBoost die tatsächliche Verspätung unterschätzt; Punkte oberhalb der Linie zeigen eine Überschätzung der tatsächlichen Verspätung.


In [ ]:
prediction_plot_data = pd.DataFrame(
    {
        "Actual": y_test,
        "Predicted": final_test_predictions,
    }
)

prediction_plot_sample_size = min(10_000, len(prediction_plot_data))
prediction_plot_sample = prediction_plot_data.sample(
    n=prediction_plot_sample_size,
    random_state=42,
)

axis_min = min(
    prediction_plot_data["Actual"].min(),
    prediction_plot_data["Predicted"].min(),
)
axis_max = max(
    prediction_plot_data["Actual"].max(),
    prediction_plot_data["Predicted"].max(),
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    prediction_plot_sample["Actual"],
    prediction_plot_sample["Predicted"],
    alpha=0.3,
    s=10,
    label="Testbeobachtungen",
)
ax.plot(
    [axis_min, axis_max],
    [axis_min, axis_max],
    linestyle="--",
    label="Ideale Prognose (Ist = Prognose)",
)
ax.set_title(
    "Tatsächliche vs. vorhergesagte Verspätung im Testmonat August – Gesamtbereich"
)
ax.set_xlabel("Tatsächliche Verspätung (Minuten)")
ax.set_ylabel("Vorhergesagte Verspätung (Minuten)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


Die zweite Darstellung begrenzt ausschließlich den sichtbaren Wertebereich auf Beobachtungen bis zum 99. Perzentil der tatsächlichen Verspätung. Die darunterliegenden Testmetriken und die finale Evaluation basieren weiterhin auf allen August-Beobachtungen. Der Zoom dient nur dazu, die Prognosequalität im typischen Wertebereich besser sichtbar zu machen.


In [ ]:
actual_p99 = y_test.quantile(0.99)
print(f"99th percentile actual delay: {actual_p99:.3f} minutes")

prediction_plot_zoom_sample = prediction_plot_sample.loc[
    prediction_plot_sample["Actual"] <= actual_p99
].copy()

zoom_axis_min = min(
    prediction_plot_zoom_sample["Actual"].min(),
    prediction_plot_zoom_sample["Predicted"].min(),
)
zoom_axis_max = max(
    actual_p99,
    prediction_plot_zoom_sample["Predicted"].quantile(0.99),
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    prediction_plot_zoom_sample["Actual"],
    prediction_plot_zoom_sample["Predicted"],
    alpha=0.3,
    s=10,
    label="Testbeobachtungen bis zum 99. Perzentil",
)
ax.plot(
    [zoom_axis_min, zoom_axis_max],
    [zoom_axis_min, zoom_axis_max],
    linestyle="--",
    label="Ideale Prognose (Ist = Prognose)",
)
ax.set_xlim(zoom_axis_min, zoom_axis_max)
ax.set_ylim(zoom_axis_min, zoom_axis_max)
ax.set_title(
    "Tatsächliche vs. vorhergesagte Verspätung im Testmonat August – Zoom bis zum 99. Perzentil"
)
ax.set_xlabel("Tatsächliche Verspätung (Minuten)")
ax.set_ylabel("Vorhergesagte Verspätung (Minuten)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Zusammenfassung der finalen Evaluation

Die Tabelle dokumentiert die finale Prognoseleistung des zuvor festgelegten XGBoost-Modells auf dem bislang unangetasteten Testmonat August 2026. Die Baselines dienen als Referenz. Die Testwerte werden nicht für weitere Anpassungen des Modells verwendet.
